# 05 — Content-based Filtering trên thể loại phim
MovieLens 1M chỉ có tên phim và thể loại, **không có cốt truyện/diễn viên/đạo diễn**.

1. Mỗi thể loại là một token: giữ nguyên Sci-Fi, Film-Noir, Children's.
2. Fit TF-IDF trên metadata các phim **trong train**; chuẩn hóa L2.
3. Hồ sơ user = tổng vector phim đã đánh giá với trọng số max(rating−3,0), sau đó chuẩn hóa. Rating 4 đóng góp 1, rating 5 đóng góp 2.
4. Cosine giữa hồ sơ và vector phim → điểm xếp hạng. Loại toàn bộ lịch sử train.

Tìm tên phim sử dụng chỉ mục TF-IDF riêng (title + genres), không ảnh hưởng đánh giá cá nhân hóa. N-gram không phải tham số để tune mô hình thể loại. Thiếu tín hiệu → popularity; hòa điểm → popularity rồi ID. Code dùng lại ở src/recommenders.py.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src/recommenders.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.recommenders import (load_data, split_validation, Popularity,
                              CollaborativeFiltering, ContentBased, evaluate, fingerprint)
from src.experiments import save_results
train, test, movies = load_data(ROOT)
print('Train:', train.shape, '| Test:', test.shape)
print('Giữ nguyên split 80/20 của notebook 01; dùng movie_id gốc.')
OUT = ROOT / 'reports/tables'
OUT.mkdir(parents=True, exist_ok=True)


Train: (800167, 4) | Test: (200042, 4)
Giữ nguyên split 80/20 của notebook 01; dùng movie_id gốc.


## Quy ước đánh giá chung với 02/04/thu_nghiem
Rating ≥ 4 là tích cực. Candidate là phim trong train, bỏ **mọi phim user đã đánh giá trong train** kể cả rating thấp. Lọc heldout theo user/movie thuộc train rồi chỉ đánh giá user có ít nhất 1 phim tích cực. Precision chia K=10, macro-average theo user; F1 tính từng user rồi lấy trung bình; NDCG dùng relevance nhị phân. Báo số dòng/user được đánh giá và số dòng bị loại.

Evaluator mới phá hòa theo popularity rồi MovieID tăng dần để tái lập; khi nhiều điểm bằng nhau, kết quả có thể khác nhẹ các notebook cũ dùng argsort/sort_values. Baseline được tính lại trong cùng pipeline. Không gọi số liệu notebook cũ là kết quả chạy mới. Đây là random holdout, không phải đánh giá dự đoán tương lai theo thời gian.


In [2]:
content = ContentBased().fit(train, movies)
baseline = Popularity().fit(train)
rows = []
for name, model in [('Popularity', baseline), ('Content-based Genres', content)]:
    metrics, per_user = evaluate(model, test, k=10)
    rows.append({'model': name, **metrics})
    per_user.to_csv(OUT / ('05_content_per_user.csv' if model is content else '05_popularity_per_user.csv'), index=False)
    print(name, metrics, flush=True)
results = pd.DataFrame(rows)
display(results)
save_results(ROOT, '05_test_results', results, train, test,
             'Fixed genre TF-IDF model. Full outer test; no test-based parameter tuning.')


Popularity {'Precision@10': 0.1518018018018018, 'Recall@10': 0.09066595053917328, 'F1@10': 0.0917778988255254, 'NDCG@10': 0.17657381251745477, 'evaluated_users': 5994, 'heldout_rows': 200042, 'evaluated_rows': 200016, 'dropped_rows': 26}
Content-based Genres {'Precision@10': 0.06771771771771773, 'Recall@10': 0.04337015120861829, 'F1@10': 0.0432603867233017, 'NDCG@10': 0.08541668063014682, 'evaluated_users': 5994, 'heldout_rows': 200042, 'evaluated_rows': 200016, 'dropped_rows': 26}


,model,Precision@10,Recall@10,F1@10,NDCG@10,evaluated_users,heldout_rows,evaluated_rows,dropped_rows
0,Popularity,0.151802,0.090666,0.091778,0.176574,5994,200042,200016,26
1,Content-based Genres,0.067718,0.043370,0.043260,0.085417,5994,200042,200016,26


In [3]:
def show(recommendations):
    frame = pd.DataFrame(recommendations, columns=['movie_id', 'score'])
    display(frame.merge(movies, on='movie_id', how='left'))
user = int(train.user_id.iloc[0])
print('Gợi ý cho user:', user)
show(content.ranked(content.scores(user), 10, content.seen[user]))
print('Tìm tên phim:')
show(content.search('Toy Story', 10))
print('Phim có thể loại tương tự:')
show(content.similar(int(train.movie_id.iloc[0]), 10))


Gợi ý cho user: 2507


,movie_id,score,title,genres
0,1968,0.864542,"Breakfast Club, The (1985)",Comedy|Drama
1,2324,0.864542,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama
2,1276,0.864542,Cool Hand Luke (1967),Comedy|Drama
3,3526,0.864542,Parenthood (1989),Comedy|Drama
4,3317,0.864542,Wonder Boys (2000),Comedy|Drama
5,1958,0.864542,Terms of Endearment (1983),Comedy|Drama
6,3083,0.864542,All About My Mother (Todo Sobre Mi Madre) (1999),Comedy|Drama
7,3591,0.864542,Mr. Mom (1983),Comedy|Drama
8,232,0.864542,Eat Drink Man Woman (1994),Comedy|Drama
9,1476,0.864542,Private Parts (1997),Comedy|Drama


Tìm tên phim:


,movie_id,score,title,genres
0,1,0.725240,Toy Story (1995),Animation|Children's|Comedy
1,3114,0.630733,Toy Story 2 (1999),Animation|Children's|Comedy
2,2966,0.166831,"Straight Story, The (1999)",Drama
3,898,0.159695,"Philadelphia Story, The (1940)",Comedy|Romance
4,2852,0.156666,"Soldier's Story, A (1984)",Drama
5,2804,0.154238,"Christmas Story, A (1983)",Comedy|Drama
6,2108,0.145524,L.A. Story (1991),Comedy|Romance
7,2961,0.144730,"Story of Us, The (1999)",Comedy|Drama
8,844,0.143310,"Story of Xinghua, The (1993)",Drama
9,2161,0.141207,"NeverEnding Story, The (1984)",Adventure|Children's|Fantasy


Phim có thể loại tương tự:


,movie_id,score,title,genres
0,3448,1.000000,"Good Morning, Vietnam (1987)",Comedy|Drama|War
1,946,1.000000,To Be or Not to Be (1942),Comedy|Drama|War
2,516,1.000000,Renaissance Man (1994),Comedy|Drama|War
3,1294,0.928407,M*A*S*H (1970),Comedy|War
4,1256,0.928407,Duck Soup (1933),Comedy|War
5,1078,0.928407,Bananas (1971),Comedy|War
6,1184,0.928407,Mediterraneo (1991),Comedy|War
7,157,0.928407,Canadian Bacon (1994),Comedy|War
8,1445,0.928407,McHale's Navy (1997),Comedy|War
9,473,0.928407,In the Army Now (1994),Comedy|War


## Hạn chế và phân tích
Các phim cùng tổ hợp thể loại có vector giống nhau, nên thứ tự có thể phụ thuộc popularity phá hòa. Không diễn giải similarity thành rating hoặc xác suất. Content-based thể loại dễ giải thích và dùng được không cần láng giềng user, nhưng thiếu đặc trưng chi tiết để phân biệt phim cùng thể loại. Không được kết luận chất lượng tương đương mô hình dùng mô tả/embedding. User mới có thể dùng tìm kiếm tên/thể loại; chưa có đánh giá cá nhân hóa cho cold-start trong bảng này.